In [0]:
last_payment_table = dbutils.widgets.get("last_payment_table")
transactions_history_table = dbutils.widgets.get("transactions_history_table")

In [0]:
display(
spark.sql(f"""   
MERGE INTO {last_payment_table} AS tgt
USING (
  SELECT ClaimNumber, EntryDate
  FROM (
    SELECT 
      ClaimNumber, 
      EntryDate,
      ROW_NUMBER() OVER (PARTITION BY ClaimNumber ORDER BY EntryDate DESC) AS rnb
    FROM {transactions_history_table}
    WHERE TransType = 'Payment' 
      AND Loaddate = date_sub(current_date(), 1)
  )
  WHERE rnb = 1
) AS Src
ON tgt.claim_number = Src.ClaimNumber
WHEN MATCHED AND tgt.entry_date <> Src.EntryDate THEN
  UPDATE SET 
    tgt.entry_date = Src.EntryDate, 
    tgt.update_date = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (claim_number, entry_date, update_date)
  VALUES (Src.ClaimNumber, Src.EntryDate, current_timestamp())
""")
)